In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Notebook 03 — Comparison & Final Summary

This notebook compares the results from:
- Notebook 01 (Naïve / Leaky Pipeline)
- Notebook 02 (Leakage-Safe Pipeline)

The goal is to show how data leakage and bias inflate performance
and how fixing them reveals the true difficulty of the task.

In [3]:
import pandas as pd

# Corrected metrics from all notebooks
results = pd.DataFrame({
    "Pipeline": [
        "Naïve (Leaky)",
        "Leakage-Safe (Random Split)",
        "Leakage-Safe (Temporal Split)"
    ],
    "ROC-AUC": [
        0.7102,
        0.7489,
        0.7374
    ],
    "Recall (Defaulters)": [
        0.0004,   # ~2 / 4965 defaulters
        0.0125,
        0.0179
    ]
})

results



,Pipeline,ROC-AUC,Recall (Defaulters)
0,Naïve (Leaky),0.7102,0.0004
1,Leakage-Safe (Random Split),0.7489,0.0125
2,Leakage-Safe (Temporal Split),0.7374,0.0179


In [4]:
results.style.format({
    "Accuracy": "{:.3f}",
    "ROC-AUC": "{:.3f}",
    "Recall (Defaulters)": "{:.3f}"
})


,Pipeline,ROC-AUC,Recall (Defaulters)
0,Naïve (Leaky),0.710,0.000
1,Leakage-Safe (Random Split),0.749,0.013
2,Leakage-Safe (Temporal Split),0.737,0.018


## Experimental Setup & Data Summary

- Dataset: Home Credit Default Risk (application_train.csv)
- Samples: 307,511 applicants
- Target distribution:
  - Non-defaulters (0): ~92%
  - Defaulters (1): ~8%
- Task: Binary classification — predict loan default at application time

This severe class imbalance makes evaluation sensitive to metric choice and leakage.


## Quantitative Results

| Pipeline                         | ROC-AUC | Recall (Defaulters) |
|----------------------------------|--------:|--------------------:|
| Naïve (Leaky)                    | 0.710   | ~0.000              |
| Leakage-Safe (Random Split)      | 0.749   | 0.013               |
| Leakage-Safe (Temporal Split)    | 0.737   | 0.018               |

Accuracy is intentionally omitted as it exceeds 90% for all pipelines
due to majority-class dominance and does not reflect model usefulness.


## Core Observations

1. The naïve (leaky) pipeline achieves a ROC-AUC of ~0.71 while identifying
   almost no defaulters, demonstrating that reasonable ranking performance
   does not imply actionable classification.

2. Removing leakage leads to modest changes in ROC-AUC but a substantial
   relative increase in recall, indicating a shift in model behavior rather
   than raw performance.

3. Temporal evaluation consistently produces lower scores than random splits,
   confirming that random evaluation is optimistically biased.

4. The small absolute differences in ROC-AUC suggest that the problem is
   intrinsically difficult, not merely poorly evaluated.


## Methodological Takeaways

- Accuracy is an inappropriate primary metric for imbalanced risk problems.
- ROC-AUC must be complemented with class-specific metrics such as recall.
- Pipeline-based preprocessing is essential to prevent silent leakage.
- Fixing evaluation methodology has greater impact on trustworthiness than
  changing the learning algorithm.


## Final Research Conclusion

This project demonstrates that data leakage and evaluation bias can produce
misleadingly confident models without improving real-world decision quality.

By systematically inducing, detecting, and correcting leakage, we show that
honest evaluation reveals the true difficulty of credit-risk prediction.

A model that performs worse under leakage-free evaluation is more valuable
than one that performs better due to flawed methodology.


## One-Line Research Claim

Leakage affects the credibility of model evaluation more than the magnitude of reported performance.
